In [1]:
#All my imports go here
import pandas as pd
import numpy as np

from density_app.functions.data_files import batchProcessFiles, file_path_traverse
from fnmatch import fnmatch
from density_app.functions.fitting_and_error import linear_fit_data_with_error, density_error, get_parabolic_fit, get_parabola_extrema, get_density_error_from_parabolic_fit, get_extended_xs
from density_app.functions.alkali_density_calculation import get_main_prefactor, get_first_order_terms_prefactor, get_second_order_terms_prefactor, get_first_order_terms, get_second_order_terms, rb_density_second_order
from density_app.functions.formatting import formatter
from density_app.functions.conversions import convertTtoKelvin

import matplotlib.pyplot as plt

from density_app.functions.constants import ELECTRON_MASS, ELECTRON_CHARGE, PLANKS_CONSTANT, BOLTZMANN_CONSTANT, BOHR_MAGNETON, LIGHT_SPEED

In [2]:
#This module is creating the dataset of densities and corresponding detunings

def get_slope(root):
    data_file_str = '*_processed*'
    experiment_file_str = '*Experiment_Params*' 
    #traverse the root directory to collect wavelengths
    #get list of all csv files
    files = file_path_traverse(root)
    #create an empty dataframe where we will store wavelength, slope, and error
    slope_data = pd.DataFrame({'ProbeWavelength':[], 'Slope':[], 'SlopeError':[]})

    for key, value in files.items():
            #for each key I need to create one processed file
            processed = ''
            exp = ''
            #experiment_params = {}
            for v in value:
                if fnmatch(v, data_file_str):
                    processed = v #locates the data file in the set of files indexed under the current key
                if fnmatch(v, experiment_file_str):
                    exp = v #locates the data file in the set of files indexed under the current key
                #processed_data = pd.read_csv(processed)
            experiment_params = pd.read_csv(exp)
            probe_wavelength = (experiment_params['LaserWavelength'].to_numpy()[0]*10**(-7))
            temperature = experiment_params['OvenTemperature'].to_numpy()[0]
            #get the magnetic field data (x), the rotations (y) and the error from the data file
            data = pd.read_csv(processed)
            #convert to density values
            x = data['Magnetic Field (Gauss)'].to_numpy()
            y = data['Rotation (Radians)'].to_numpy()
            error = data['Rotation Standard Deviation'].to_numpy()
            param, cov = linear_fit_data_with_error(x,y,error)
            slope = param[0]
            slope_error = np.sqrt(cov[0][0])
            #put it in a dataframe
            #calculate error 
          
            new_row = pd.DataFrame({'ProbeWavelength':[probe_wavelength], 'Slope': [slope], 'SlopeError':[slope_error]})
            slope_data = pd.concat([slope_data, new_row], ignore_index=True) 
        
    return slope_data



def get_data(data_set):
    x = data_set['ProbeWavelength'].to_numpy()
    y = data_set['Slope'].to_numpy()
    error = data_set['SlopeError'].to_numpy()
    return x,y,error
     

def get_dataset(temp):
    dir = '/Users/eleanor/Desktop/Data4Eleanor/' + f"{temp}"+"C"
    

def detuning_term(opt_path, probe, d1, d2, temp):
    coeff0 = (opt_path*ELECTRON_CHARGE**2*BOHR_MAGNETON)/(6*ELECTRON_MASS*PLANKS_CONSTANT*LIGHT_SPEED)
    coeff1 = probe**2/(3*LIGHT_SPEED**2)
    coeff2 = (PLANKS_CONSTANT*probe)/(LIGHT_SPEED*BOLTZMANN_CONSTANT*convertTtoKelvin(temp))
    term1 = ((4*d1**2)/(d1-probe)**2) + ((7*d2**2)/(d2-probe)**2) - ((2*d1*d2)/((d1-probe)*(d2-probe)))
    term2 = (d1/(d1-probe))-(d2/(d2-probe))

    f = coeff0*(coeff1*term1 - coeff2*term2)
    return f

    

dir_80 = '/Users/eleanor/Desktop/Data4Eleanor/80C'
dir_90 = '/Users/eleanor/Desktop/Data4Eleanor/90C'
dir_110 = '/Users/eleanor/Desktop/Data4Eleanor/110C'
dir_120 = '/Users/eleanor/Desktop/Data4Eleanor/120C'

slope_data_80C = get_slope(dir_80)
slope_data_90C = get_slope(dir_90)
slope_data_110C = get_slope(dir_110)
slope_data_120C = get_slope(dir_120)

#print(get_data(data_80C))
#print(density_80C['ProbeWavelength'].to_numpy())

In [3]:
#THIS IS FOR CELL 314A
cell = '314A'
d2_orig = 7.80034e-5
d2_adjust = 0#-0.00009e-5
d1_resonance = 7.94768e-5 #cm
d2_resonance = d2_orig+d2_adjust #cm
optical_path = 3.7 #cm
verdet_adjustment = 0 #we do not want to include any adjustment right now

wl_error = 0#.00003e-5
d2_error = 0#.00003e-5
d1_error = 0#.00003e-5

length = 3.7
length_error = 0#.005

In [42]:
#we now have data sets by temperature that contain [wavelength, slope, slope error]

wl, slope, serr = get_data(slope_data_90C)

#print(len(wl))

#Slope difference method
f0 = detuning_term(optical_path, wl[0], d1_resonance, d2_resonance, 120)
f1 = detuning_term(optical_path, wl[1], d1_resonance, d2_resonance, 120)
f2 = detuning_term(optical_path, wl[2], d1_resonance, d2_resonance, 120)
f3 = detuning_term(optical_path, wl[3], d1_resonance, d2_resonance, 120)
f4 = detuning_term(optical_path, wl[4], d1_resonance, d2_resonance, 120)
f5 = detuning_term(optical_path, wl[5], d1_resonance, d2_resonance, 120)

sd01 = slope[0]-slope[1]
sd12 = slope[1]-slope[2]
sd23 = slope[2]-slope[3]
sd34 = slope[3]-slope[4]
sd45 = slope[4]-slope[5]

sd05 = slope[0]-slope[5]
sd15 = slope[1]-slope[5]
sd25 = slope[2]-slope[5]

max_index = 6
temp = 90
rbs = []
for i in range (0,max_index):
    for j in range(i+1, max_index):
        diff_s = slope[i]-slope[j]
        diff_f = detuning_term(optical_path, wl[i], d1_resonance, d2_resonance, temp)-detuning_term(optical_path, wl[j], d1_resonance, d2_resonance, temp)
        rbs.append(diff_s/diff_f)
        #print(i,j,diff_s)

print(np.average(np.array(rbs)))
print(np.std(np.array(rbs)))



3684785561798.0864
1487432616863.2393


| Temperature | Density, Original Flavor                | Percent Error | Density, Alternate Version | Percent Difference |
| ----------- | --------------------------------------- | ------------- |-------------------------- |------------------- |
|          80C| $1.63\times10^{12}\pm 6.8\times10^{10}$ |            4% |      $1.81\times10^{12}$ |               +12% | 
|          90C| $3.32\times10^{12}\pm 7.0\times10^{10}$ |            2% |      $3.68\times10^{12}$ |               +11% |
|         110C| $1.01\times10^{13}\pm 5.4\times10^{11}$ |            5% |      $1.04\times10^{13}$ |                +3% |
|         120C| $1.94\times10^{13}\pm 3.2\times10^{11}$ |            2% |      $2.05\times10^{13}$ |                +6% |

However, the standard deviations in the new method, which I was using as a proxy for error, are pretty large, so I think that the original method is going to be much more reliable overall unfortunately 